In [ ]:
!pip install sae-lens transformer-lens

In [ ]:
!pip install "numpy<2.0.0"

In [2]:
import os
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
import shap
import matplotlib.pyplot as plt

# Requires: pip install sae-lens transformer-lens xgboost kagglehub
from transformer_lens import HookedTransformer
from sae_lens import SAE
import kagglehub
from kagglehub import KaggleDatasetAdapter
from google.colab import userdata
from huggingface_hub import login


from huggingface_hub import login
import gc

torch.set_grad_enabled(False)

# Paste your HF token here
HF_TOKEN = ""

login(token=HF_TOKEN)
print("Successfully logged into Hugging Face")



def load_and_prepare_kaggle_data(sample_size=500):
   
    print("Loading DAIGT V2 Dataset via Kagglehub...")
    file_path = "train_v2_drcat_02.csv"
    try:
        df = kagglehub.load_dataset(
            KaggleDatasetAdapter.PANDAS,
            "thedrcat/daigt-v2-train-dataset",
            file_path
        )
    except Exception as e:
        print(f"Error loading Kaggle dataset: {e}")
        return None

    df = df.dropna(subset=['text', 'label'])
    df_human = df[df['label'] == 0]
    df_ai = df[df['label'] == 1]

    half_size = sample_size // 2
    if len(df_human) > half_size: df_human = df_human.sample(half_size, random_state=42)
    if len(df_ai) > half_size: df_ai = df_ai.sample(half_size, random_state=42)

    df_balanced = pd.concat([df_human, df_ai]).sample(frac=1, random_state=42).reset_index(drop=True)
    return df_balanced


def setup_mechanistic_microscope():
    print("\n--- Loading Proxy Model & SAE ---")
    
    
    torch.cuda.empty_cache()
    gc.collect()

    n_gpus = torch.cuda.device_count()
    print(f"Found {n_gpus} GPUs. Sharding model...")

    # Load Gemma-2-2B split across both T4s
    model = HookedTransformer.from_pretrained(
        "gemma-2-2b", 
        n_devices=n_gpus,
        dtype=torch.float16
    )

    # Find exactly which GPU Layer 20 was placed on
    layer_20_device = next(model.blocks[20].parameters()).device
    print(f"Layer 20 was mapped to: {layer_20_device}")

    # Load the SAE directly onto the same GPU as Layer 20
    sae = SAE.from_pretrained(
        release="gemma-scope-2b-pt-res-canonical",
        sae_id="layer_20/width_16k/canonical",
        device=str(layer_20_device) 
    )
    
    return model, sae, str(layer_20_device)

def text_to_sae_latents(text, model, sae, device, max_length=128):
    # Tokenize and truncate to manage memory
    tokens = model.to_tokens(text, prepend_bos=True)
    if tokens.shape[1] > max_length:
        tokens = tokens[:, :max_length]

    hook_point = "blocks.20.hook_resid_post"
    
    # Run forward pass and cache activations
    _, cache = model.run_with_cache(tokens, names_filter=[hook_point])
    
    dense_acts = cache[hook_point]
    
    # Encode with SAE
    feature_acts = sae.encode(dense_acts)
    
    # Pool sequence
    document_vector = feature_acts.max(dim=1).values.squeeze(0)
    
    # Move to CPU for XGBoost 
    return document_vector.cpu().numpy()

def extract_all_latents(df, model, sae, device):
    print("\n--- Extracting Sparse Latents ---")
    print(f"Processing {len(df)} texts through Gemma-2-2B...")

    features = []
    for idx, row in df.iterrows():
        latents = text_to_sae_latents(row['text'], model, sae, device)
        features.append(latents)
        if (idx + 1) % 50 == 0:
            print(f"Processed {idx + 1}/{len(df)}...")

    # Create DataFrame of the 16,384 sparse features
    
    feature_cols = [f"Feature_{i}" for i in range(len(features[0]))]
    features_df = pd.DataFrame(features, columns=feature_cols)

    return features_df


def train_evaluate_latent_detector(X, y):
    print("\n--- Training Latent Detector (XGBoost) ---")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

    
    clf = XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1
    )
    clf.fit(X_train, y_train)

    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"SAE Latent Classification Accuracy: {acc:.4f}")
    print(classification_report(y_test, preds, target_names=["Human", "AI"]))

    return clf, X_train, X_test


if __name__ == "__main__":
    
    df = load_and_prepare_kaggle_data(sample_size=500)

    if df is not None:
        
        model, sae, device = setup_mechanistic_microscope()

        
        X_features = extract_all_latents(df, model, sae, device)
        y_labels = df['label']

       
        xgb_model, X_train, X_test = train_evaluate_latent_detector(X_features, y_labels)

      
        feature_importances = xgb_model.feature_importances_
        top_indices = np.argsort(feature_importances)[-10:][::-1]

        print("\n--- Top 10 Most Predictive SAE Features ---")
        for rank, idx in enumerate(top_indices):
            print(f"{rank+1}. Feature_{idx} (Importance: {feature_importances[idx]:.4f})")
            print(f"   -> Look up on Neuronpedia: https://neuronpedia.org/gemma-2-2b/20-gemmascope-res-16k/{idx}")

Successfully logged into Hugging Face!
Loading DAIGT V2 Dataset via Kagglehub...


/tmp/ipykernel_319/3312816298.py:42: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(
`torch_dtype` is deprecated! Use `dtype` instead!



--- Loading Proxy Model & SAE ---
Found 2 GPUs. Sharding model...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model gemma-2-2b into HookedTransformer
Layer 20 was mapped to: cuda:0

--- Extracting Sparse Latents ---
Processing 500 texts through Gemma-2-2B...
Processed 50/500...
Processed 100/500...
Processed 150/500...
Processed 200/500...
Processed 250/500...
Processed 300/500...
Processed 350/500...
Processed 400/500...
Processed 450/500...
Processed 500/500...

--- Training Latent Detector (XGBoost) ---
SAE Latent Classification Accuracy: 0.9800
              precision    recall  f1-score   support

       Human       0.96      1.00      0.98        50
          AI       1.00      0.96      0.98        50

    accuracy                           0.98       100
   macro avg       0.98      0.98      0.98       100
weighted avg       0.98      0.98      0.98       100


--- Top 10 Most Predictive SAE Features ---
1. Feature_8573 (Importance: 0.4261)
   -> Look up on Neuronpedia: https://neuronpedia.org/gemma-2-2b/20-gemmascope-res-16k/8573
2. Feature_6832 (Importance: 0.1470)

# with 2000 samples

In [1]:
import os
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
import shap
import matplotlib.pyplot as plt


from transformer_lens import HookedTransformer
from sae_lens import SAE
import kagglehub
from kagglehub import KaggleDatasetAdapter
from google.colab import userdata
from huggingface_hub import login


from huggingface_hub import login
import gc

torch.set_grad_enabled(False)

# Paste your HF token here
HF_TOKEN = ""

login(token=HF_TOKEN)
print("Successfully logged into Hugging Face!")


def load_and_prepare_kaggle_data(sample_size=500):

    print("Loading DAIGT V2 Dataset via Kagglehub...")
    file_path = "train_v2_drcat_02.csv"
    try:
        df = kagglehub.load_dataset(
            KaggleDatasetAdapter.PANDAS,
            "thedrcat/daigt-v2-train-dataset",
            file_path
        )
    except Exception as e:
        print(f"Error loading Kaggle dataset: {e}")
        return None

    df = df.dropna(subset=['text', 'label'])
    df_human = df[df['label'] == 0]
    df_ai = df[df['label'] == 1]

    half_size = sample_size // 2
    if len(df_human) > half_size: df_human = df_human.sample(half_size, random_state=42)
    if len(df_ai) > half_size: df_ai = df_ai.sample(half_size, random_state=42)

    df_balanced = pd.concat([df_human, df_ai]).sample(frac=1, random_state=42).reset_index(drop=True)
    return df_balanced


def setup_mechanistic_microscope():
    print("\n--- Loading Proxy Model & SAE ---")
    
    
    torch.cuda.empty_cache()
    gc.collect()

    n_gpus = torch.cuda.device_count()
    print(f"Found {n_gpus} GPUs. Sharding model...")

    # Load Gemma-2-2B split across both T4s
    model = HookedTransformer.from_pretrained(
        "gemma-2-2b", 
        n_devices=n_gpus, 
        dtype=torch.float16
    )

    # Find exactly which GPU Layer 20 was placed on
    layer_20_device = next(model.blocks[20].parameters()).device
    print(f"Layer 20 was mapped to: {layer_20_device}")

    # Load the SAE directly onto the same GPU as Layer 20
    sae = SAE.from_pretrained(
        release="gemma-scope-2b-pt-res-canonical",
        sae_id="layer_20/width_16k/canonical",
        device=str(layer_20_device) 
    )
    
    return model, sae, str(layer_20_device)

def text_to_sae_latents(text, model, sae, device, max_length=128):
    # Tokenize and truncate to manage memory
    tokens = model.to_tokens(text, prepend_bos=True)
    if tokens.shape[1] > max_length:
        tokens = tokens[:, :max_length]

    
    tokens = tokens.to(model.embed.W_E.device)

    hook_point = "blocks.20.hook_resid_post"
    
    # Run forward pass and cache activations
    _, cache = model.run_with_cache(tokens, names_filter=[hook_point])
    
    
    dense_acts = cache[hook_point].to(sae.device)
    
    # Encode with SAE
    feature_acts = sae.encode(dense_acts)
    
    # Pool sequence
    document_vector = feature_acts.max(dim=1).values.squeeze(0)
    
    # Move to CPU for XGBoost
    return document_vector.cpu().numpy()

def extract_all_latents(df, model, sae, device):
    print("\n--- Extracting Sparse Latents ---")
    print(f"Processing {len(df)} texts through Gemma-2-2B...")

    features = []
    for idx, row in df.iterrows():
        latents = text_to_sae_latents(row['text'], model, sae, device)
        features.append(latents)
        if (idx + 1) % 50 == 0:
            print(f"Processed {idx + 1}/{len(df)}...")

    
    feature_cols = [f"Feature_{i}" for i in range(len(features[0]))]
    features_df = pd.DataFrame(features, columns=feature_cols)

    return features_df


def train_evaluate_latent_detector(X, y):
    print("\n--- Training Latent Detector (XGBoost) ---")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

    
    clf = XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1
    )
    clf.fit(X_train, y_train)

    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"SAE Latent Classification Accuracy: {acc:.4f}")
    print(classification_report(y_test, preds, target_names=["Human", "AI"]))

    return clf, X_train, X_test


if __name__ == "__main__":
    
    df = load_and_prepare_kaggle_data(sample_size=2000)

    if df is not None:
        
        model, sae, device = setup_mechanistic_microscope()

      
        X_features = extract_all_latents(df, model, sae, device)
        y_labels = df['label']

      
        xgb_model, X_train, X_test = train_evaluate_latent_detector(X_features, y_labels)

      
        feature_importances = xgb_model.feature_importances_
        top_indices = np.argsort(feature_importances)[-10:][::-1]

        print("\n--- Top 10 Most Predictive SAE Features ---")
        for rank, idx in enumerate(top_indices):
            print(f"{rank+1}. Feature_{idx} (Importance: {feature_importances[idx]:.4f})")
            print(f"   -> Look up on Neuronpedia: https://neuronpedia.org/gemma-2-2b/20-gemmascope-res-16k/{idx}")

2026-02-26 08:43:18.980241: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772095399.006910     400 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772095399.015936     400 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772095399.036248     400 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772095399.036265     400 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772095399.036268     400 computation_placer.cc:177] computation placer alr

Successfully logged into Hugging Face!
Loading DAIGT V2 Dataset via Kagglehub...


/tmp/ipykernel_400/880307408.py:42: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(
`torch_dtype` is deprecated! Use `dtype` instead!



--- Loading Proxy Model & SAE ---
Found 2 GPUs. Sharding model...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model gemma-2-2b into HookedTransformer
Layer 20 was mapped to: cuda:0

--- Extracting Sparse Latents ---
Processing 2000 texts through Gemma-2-2B...
Processed 50/2000...
Processed 100/2000...
Processed 150/2000...
Processed 200/2000...
Processed 250/2000...
Processed 300/2000...
Processed 350/2000...
Processed 400/2000...
Processed 450/2000...
Processed 500/2000...
Processed 550/2000...
Processed 600/2000...
Processed 650/2000...
Processed 700/2000...
Processed 750/2000...
Processed 800/2000...
Processed 850/2000...
Processed 900/2000...
Processed 950/2000...
Processed 1000/2000...
Processed 1050/2000...
Processed 1100/2000...
Processed 1150/2000...
Processed 1200/2000...
Processed 1250/2000...
Processed 1300/2000...
Processed 1350/2000...
Processed 1400/2000...
Processed 1450/2000...
Processed 1500/2000...
Processed 1550/2000...
Processed 1600/2000...
Processed 1650/2000...
Processed 1700/2000...
Processed 1750/2000...
Processed 1800/2000...
Processed 1850/2000...
P